# 04 -- LLM self-play RL (Step 7)

Warm-start from the SFT checkpoint (structured + masked, step 25,000) and keep improving it by self-play: each
**round** plays `games_per_round` games against itself (moves sampled from its own masked move probabilities),
gives every move its game's score (the result; with `end_scoring: stockfish`, a game that ends by
repetition, the 50-move rule, or the ply cap is scored by Stockfish's judgment of its final position), and
makes one update pass (policy gradient with the value head as baseline,
a KL pull toward the SFT model, and a value loss). Then a few kept checkpoints are rated into
`results/results.csv`, continuing the SFT run's Elo-vs-compute curve (x-axis = SFT's 7.69 GPU-h + RL GPU-h).

Settings: `configs/llm_rl.yaml` (provisional defaults -- revisit after the test run), `configs/base.yaml`,
`configs/eval.yaml`. The player's settings come from the SFT checkpoint itself.

**First run `SMOKE_TEST = True`** (2 tiny rounds, separate run folder, no eval): checks the whole loop, memory,
and timing. Then set it to `False`; `MAX_ROUNDS` limits how many rounds this session plays (the run resumes
from where it stopped next time).

**Safe to interrupt.** Every round is saved (resume state on the latest round); rerunning continues the run.

Headless on the pod (in tmux):
`jupyter nbconvert --to notebook --execute notebooks/04_train_llm_rl.ipynb --output-dir /workspace/chess-ai/logs --ExecutePreprocessor.timeout=-1`

**Needs a GPU** (L4 pod).

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
# Before each session, in a pod terminal (code is never edited on the pod):
#   cd /workspace/Chess_Bot && git fetch && git checkout BRANCH && git reset --hard origin/BRANCH
import os
REPO_DIR = "/workspace/Chess_Bot"
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import yaml
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
import torch
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("GPU:", GPU, "| CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters
Everything comes from `configs/llm_rl.yaml` except the three flags. The run folder is named after the settings
that define the run, so rerunning with the same settings resumes it.

In [ ]:
import dataclasses
import json

from train.llm_rl import RLConfig, list_checkpoints, run_name, start_policies, train

RUN_TRAINING = True   # False = skip straight to the curves and the evaluation
SMOKE_TEST = False    # True = 2 tiny rounds in a separate "_smoke" folder (checks the loop, memory, timing); no eval
MAX_ROUNDS = None     # rounds to play this session (None = up to cfg.rounds); the run resumes later

base_cfg = yaml.safe_load(open("configs/base.yaml"))
CFG = RLConfig.from_dict(yaml.safe_load(open("configs/llm_rl.yaml")))
if SMOKE_TEST:
    CFG = dataclasses.replace(CFG, rounds=2, games_per_round=2, max_plies=40, step_positions=32, checkpoint_every=1)
MAX_GPU_HOURS = base_cfg.get("max_gpu_hours_per_arm")   # safety cap (null = none yet; decided later)

sft_step_dir = os.path.join(ROOT, "checkpoints", "sft", CFG.init_run, "checkpoints", f"step_{CFG.init_step:06d}")
from model.llm_policy import PolicyConfig
POLICY = PolicyConfig.from_dict(json.load(open(os.path.join(sft_step_dir, "policy_config.json"))))
RUN = run_name(POLICY, CFG) + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = os.path.join(ROOT, "checkpoints", "llm_rl", RUN)
print("start:", sft_step_dir)
print("player:", POLICY.representation, "| masked" if POLICY.masked else "| unmasked")
print("RL:", CFG)
print("run folder:", RUN_DIR, "| safety cap:", MAX_GPU_HOURS, "GPU-h")

## Load the models
Two copies of Qwen3-0.6B with the SFT adapters: the one being trained, and a frozen reference for the KL
penalty (~1.2 GB each). On a resume, the trained one's weights are replaced by the latest round's.

In [ ]:
if RUN_TRAINING:
    policy, ref, INIT_GPU_HOURS = start_policies(CFG, ROOT)
    n_train = sum(p.numel() for p in policy.trainable_parameters())
    print(f"dtype {policy.model.get_input_embeddings().weight.dtype} on {policy.device} | trainable {n_train / 1e6:.1f}M")
    print(f"GPU-hours before RL (SFT run at step {CFG.init_step}): {INIT_GPU_HOURS:.2f}")
else:
    print("RUN_TRAINING is False -- skipping model load")

## Train
One line per round: W/D/L (White/draw/Black), average game length, policy entropy (how spread out its move
choices are -- a crash toward 0 means collapse), KL to SFT, value loss, and where the time went.

In [ ]:
if RUN_TRAINING:
    torch.cuda.reset_peak_memory_stats()
    summary = train(policy, ref, CFG, RUN_DIR, init_gpu_hours=INIT_GPU_HOURS, max_gpu_hours=MAX_GPU_HOURS,
                    max_rounds=MAX_ROUNDS, extra_info={"gpu": GPU, "cpu": CPU_MODEL, "sft_checkpoint": sft_step_dir})
    print(json.dumps(summary, indent=2))
    import pandas as pd
    log = pd.read_csv(os.path.join(RUN_DIR, "rl_log.csv"))
    per_round = (log["selfplay_seconds"] + log["update_seconds"]).mean()
    print(f"\n{per_round / 60:.1f} min/round ({log['selfplay_seconds'].mean() / 60:.1f} self-play + "
          f"{log['update_seconds'].mean() / 60:.1f} update) | peak GPU memory {summary.get('peak_gpu_memory_gb')} GB")
    if SMOKE_TEST:
        full = RLConfig.from_dict(yaml.safe_load(open("configs/llm_rl.yaml")))
        moves_per_game = log["positions"].sum() / log["games"].sum()
        print(f"smoke: {moves_per_game:.0f} moves/game at max_plies={CFG.max_plies}; real rounds play "
              f"{full.games_per_round} games up to {full.max_plies} plies -- scale the timing from that")
    del policy, ref
    torch.cuda.empty_cache()
else:
    print("RUN_TRAINING is False -- skipping training")

## Training curves
What to look for (the Step 7 gate is "no collapse, Elo flat or rising"): entropy drifting down slowly is
normal, a crash toward 0 is collapse; KL should stay small and bounded; value loss should fall as the value
head learns to predict self-play results; decisive games (not draws) are where the learning signal is.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(os.path.join(RUN_DIR, "rl_log.csv"))
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0, 0].stackplot(log["round"], log["white_wins"], log["draws"], log["black_wins"], labels=["White wins", "draws", "Black wins"])
ax[0, 0].set_title("self-play results per round"); ax[0, 0].legend(loc="lower left")
ax[0, 1].plot(log["round"], log["entropy"], "o-"); ax[0, 1].set_title("policy entropy (nats)")
ax[0, 2].plot(log["round"], log["kl"], "o-"); ax[0, 2].set_title("KL to the SFT model")
ax[1, 0].plot(log["round"], log["value_loss"], "o-"); ax[1, 0].set_title("value loss")
ax[1, 1].plot(log["round"], log["avg_plies"], "o-", label="avg plies")
ax[1, 1].plot(log["round"], log["max_plies_draws"], "o-", label="games hitting the ply cap")
for col, label in [("ends_repetition", "ended by repetition"), ("ends_fifty_moves", "ended by 50-move rule"),
                   ("rescored_lopsided", "re-scored, lopsided (<0.3 or >0.7)")]:
    if col in log:   # logged from Oct 18 (run 2) on
        ax[1, 1].plot(log["round"], log[col], "o-", label=label)
ax[1, 1].legend(fontsize=8)
ax[1, 1].set_title("game length and how games ended")
ax[1, 2].plot(log["round"], log["train_gpu_hours"], "o-"); ax[1, 2].set_title("total training GPU-hours (SFT + RL)")
for a in ax.flat:
    a.set_xlabel("round")
plt.tight_layout()
plt.show()
print(log.tail(10).to_string(index=False))

## Evaluate checkpoints into `results.csv`

Rate 3-4 kept checkpoints spread over the run (by default: up to 4, evenly spaced) against the 2-3 reference
members nearest their strength, exactly like the SFT checkpoints in `02`. Each row's `train_gpu_hours` is SFT's
7.69 + the RL hours at that round, and `label_cpu_hours` is charged because the run started from SFT. A
checkpoint that already has its full game count is skipped on a rerun. Skipped in a smoke test.

In [ ]:
from eval.anchors import ResultsTable, build_reference_agents, close_agents
from eval.elo import format_table
from eval.match import evaluate_agent
from train.llm_rl import gpu_hours_at
from train.sft import load_checkpoint

EVAL_ROUNDS = None   # e.g. [5, 10, 20]; None = up to 4 of the every-checkpoint_every-rounds checkpoints, evenly spaced
OPPONENTS = ["greedy", "step_20000", "sf_1320"]   # move up (sf_1320, sf_1500) if the model gets stronger

kept = dict(list_checkpoints(RUN_DIR))
print("kept rounds:", sorted(kept))
if EVAL_ROUNDS is None and kept:
    rounds = [r for r in sorted(kept) if r % CFG.checkpoint_every == 0] or sorted(kept)   # r5, r10, ... (not a stray last round)
    EVAL_ROUNDS = sorted({rounds[round(i * (len(rounds) - 1) / 3)] for i in range(4)}) if len(rounds) > 4 else rounds

if SMOKE_TEST:
    print("smoke test -- skipping evaluation")
else:
    eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
    results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
    FIXED = results.fixed()
    print("fixed ratings:", {n: round(v) for n, v in sorted(FIXED.items(), key=lambda kv: kv[1])})
    print("evaluating rounds:", EVAL_ROUNDS)
    GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
    opponents = build_reference_agents(results, OPPONENTS, ROOT)
    try:
        for rnd in EVAL_ROUNDS:
            name = f"rl_{RUN}@r{rnd}"
            if results.role(name) == "model" and int(results.rows[name]["games"]) >= GAMES * len(OPPONENTS):
                print(f"{name}: already evaluated ({results.rows[name]['games']} games) -- skipped")
                continue
            agent = load_checkpoint(kept[rnd], name=name)

            def save(est, recs, rnd=rnd):
                results.log_model(est, recs, run=f"llm_rl/{RUN}", step=rnd, seed=CFG.seed,
                                  train_gpu_hours=gpu_hours_at(RUN_DIR, rnd),
                                  label_cpu_hours=base_cfg["label_cpu_hours"] if CFG.init_run else 0.0, gpu=GPU,
                                  note=f"self-play RL from sft {CFG.init_run}@{CFG.init_step}")
                results.save()
                print(format_table([est]))

            evaluate_agent(agent, opponents, FIXED, GAMES, batch_games=BATCH, on_batch=save)
            del agent
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)
    display(pd.read_csv(results.path))

## Elo vs. compute
Every run in `results.csv` on one chart: SFT checkpoints (`step_20000` as a hollow marker), each RL run in its
own color joined to the SFT checkpoint it started from, 95% CIs, nearby reference ratings as dashed lines.
Saved to `results/elo_vs_compute.png` on the volume (`eval/plots.py`, reads only `results.csv`).

In [ ]:
from eval.plots import plot_elo_vs_compute

RESULTS_FILE = os.path.join(ROOT, yaml.safe_load(open("configs/eval.yaml"))["results_file"])
PLOT_FILE = os.path.join(ROOT, "results", "elo_vs_compute.png")
fig = plot_elo_vs_compute(RESULTS_FILE, PLOT_FILE)
display(fig)
print("saved:", PLOT_FILE)